In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 读取数据
print("Loading data...")
data = pd.read_csv('melb_data.csv')

y = data.Price

# 为了简化问题，我们只采用数值列作为特征进行训练
melb_predictors = data.drop(['Price'], axis=1)
x = melb_predictors.select_dtypes(exclude=['object'])

# 将原始数据划分为训练集和验证集
x_train, x_valid, y_train, y_valid = train_test_split(x, y, train_size=0.8, test_size=0.2, random_state=0)

print("Setup complete")

Loading data...
Setup complete


In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# 我们利用 MAE 检验三种处理方法的好坏
def score_dataset(x_train, x_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=10, random_state=0)
    model.fit(x_train, y_train)
    preds = model.predict(x_valid)
    return mean_absolute_error(y_valid, preds)

# 1. 直接删除含有缺失值的列
cols_with_missing = [col for col in x_train.columns if x_train[col].isnull().any()]

reduced_x_train = x_train.drop(cols_with_missing, axis=1)
reduced_x_valid = x_valid.drop(cols_with_missing, axis=1)

print("MAE from Approach 1 (Drop columns with missing values):")
print(score_dataset(reduced_x_train, reduced_x_valid, y_train, y_valid))

# 2. 采用填充均值的方法填补缺失数据列
from sklearn.impute import SimpleImputer

my_imputer = SimpleImputer()
imputed_x_train = pd.DataFrame(my_imputer.fit_transform(x_train))
imputed_x_valid = pd.DataFrame(my_imputer.transform(x_valid))

imputed_x_train.columns = x_train.columns
imputed_x_valid.columns = x_valid.columns

print("MAE from Approach 2 (Imputation):")
print(score_dataset(imputed_x_train, imputed_x_valid, y_train, y_valid))

# 3. 填补缺失列，同时记录哪些列缺失
x_train_plus = x_train.copy()
x_valid_plus = x_valid.copy()

for col in cols_with_missing:
    x_train_plus[col + '_was_missing'] = x_train_plus[col].isnull()
    x_valid_plus[col + '_was_missing'] = x_valid_plus[col].isnull()

my_imputer = SimpleImputer()
imputed_x_train_plus = pd.DataFrame(my_imputer.fit_transform(x_train_plus))
imputed_x_valid_plus = pd.DataFrame(my_imputer.transform(x_valid_plus))

imputed_x_train_plus.columns = x_train_plus.columns
imputed_x_valid_plus.columns = x_valid_plus.columns

print("MAE from Approach 3 (An Extension to Imputation):")
print(score_dataset(imputed_x_train_plus, imputed_x_valid_plus, y_train, y_valid))

MAE from Approach 1 (Drop columns with missing values):
183550.22137772635
MAE from Approach 2 (Imputation):
178166.46269899711
MAE from Approach 3 (An Extension to Imputation):
178927.503183954


In [6]:
print(x_train.shape)

# 计算含有缺失值的列的缺失数目
missing_val_count_by_column = (x_train.isnull().sum())
print(missing_val_count_by_column[missing_val_count_by_column > 0])

(10864, 12)
Car               49
BuildingArea    5156
YearBuilt       4307
dtype: int64
